# Stage 1+2 - Explore DOTA & understand tiling

This notebook is the **exploration surface**: load images, see oriented boxes
inline, and watch tiling work on a single image. The heavy logic lives in
`dota_utils.py` (imported below) so this notebook and the batch scripts share
one implementation.

Rule of thumb for this project:
- **Notebook** (here) = understand, inspect, eyeball one image.
- **`scripts/02_tile_dota.py`** = tile the *whole* dataset once.
- **`yolo` CLI / train script** = the long training run (Stage 4).

### Setup: make `dota_utils` importable
Works whether you launch Jupyter from the repo root or from `notebooks/`.

In [ ]:
import os, sys
_here = os.getcwd()
for _cand in (_here, os.path.dirname(_here), os.path.join(_here, "..")):
    if os.path.exists(os.path.join(_cand, "dota_utils.py")):
        sys.path.insert(0, os.path.abspath(_cand)); break

import numpy as np
from PIL import Image
from IPython.display import display
import dota_utils as du
print("loaded dota_utils; classes:", len(du.DOTA_CLASSES))

### Point to your data
Edit these to your extracted DOTA folders.

In [ ]:
DATA = "../DOTA/train"                 # <-- change me
IMAGES = os.path.join(DATA, "images")
LABELS = os.path.join(DATA, "labelTxt")

# pick the first image as an example
fname = sorted(f for f in os.listdir(IMAGES) if f.endswith((".png",".jpg")))[0]
img_path = os.path.join(IMAGES, fname)
lbl_path = os.path.join(LABELS, os.path.splitext(fname)[0] + ".txt")
print("example:", fname)

## Stage 1 - See the oriented boxes

DOTA labels each object as a **quadrilateral** (8 numbers = 4 corners), not the
`(x, y, w, h)` of COCO. A plane or ship viewed from above sits at an arbitrary
angle, so an axis-aligned box would be mostly empty space and would overlap its
neighbours. That's why we use an OBB model later.

In [ ]:
objs = du.parse_dota_label(lbl_path)
img = Image.open(img_path).convert("RGB")
annotated = du.draw_obbs(img, objs)
display(annotated.resize((900, int(900 * img.height / img.width))))  # shrink to view
print("first object (corners, class, difficult):", objs[0])

### The numbers that force us to tile

In [ ]:
s = du.image_label_stats(objs, *img.size)
print(f"image size      : {s['width']} x {s['height']} px")
print(f"objects         : {s['n_objects']}")
print(f"object size px  : min {s['size_min']:.0f} / median {s['size_median']:.0f} / max {s['size_max']:.0f}")
print(f"under 40 px     : {100*s['frac_under_40px']:.0f}%")
print(f"per class       : {s['per_class']}")

**Read that `under 40px` number.** If you resized this image to 640x640 to
feed a network, anything under ~40px becomes a handful of pixels and the
detector cannot find it. You can't tune that away - it's lost information. The
fix is to slice the image into patches at (near) native resolution.

## Stage 2 - Watch tiling work on this one image

`du.iter_tiles` is the *exact* same generator the batch script uses. Here we
just display the patches instead of writing them. Watch how an object on a
patch seam is kept whole in one patch and dropped (as a sliver) from the
neighbour - that's the `iof` threshold doing its job.

In [ ]:
arr = np.array(img)
tiles = list(du.iter_tiles(arr, objs, patch=1024, gap=200, iof_thresh=0.7))
print(f"{len(tiles)} non-empty patches from this image")

for left, top, crop, patch_objs in tiles[:3]:
    print(f"patch @ (left={left}, top={top}) -> {len(patch_objs)} objects")
    display(du.draw_obbs(crop, patch_objs).resize((420, 420)))

Boxes still hug their objects after slicing, and their coordinates are now
relative to the patch. That's what the model will train on.

### When you're happy, tile the whole dataset with the script
```bash
python scripts/02_tile_dota.py \
    --images-dir DOTA/train/images --labels-dir DOTA/train/labelTxt \
    --out-dir DOTA-tiled/train --patch 1024 --gap 200
# repeat for val
```
Next: **Stage 3** - convert these patches to YOLO-OBB format (class indices +
normalized coords) and build the train/val/**test** split.